In [20]:
import plotly.express as px

In [ ]:
import pandas as pd

import plotly.express as px

import plotly.graph_objects as po



df = pd.read_csv("../../data/raw/sample-data.csv") 

df['Timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df = df.drop(columns=['Date', 'Time'])
# print(df.head())

# Group the data by date and create a dictionary of dataframes, one for each day
df['Date'] = df['Timestamp'].dt.date
daily_dfs = {date: group.drop(columns='Date') for date, group in df.groupby('Date')}
# print(daily_dfs)

fig = px.line(df, x='Timestamp', y='Value', title='Data Chart')
fig.show()

# Get statistics for each day using .describe() on each daily dataframe
daily_stats = {date: day_df['Value'].describe() for date, day_df in daily_dfs.items()}
for date, stats in daily_stats.items():
    print(f"Statistics for {date}:\n{stats}\n")

# Plot daily statistics (mean, min, max) for each day
import plotly.graph_objects as go

dates = list(daily_stats.keys())
means = [daily_stats[d]['mean'] for d in dates]
mins = [daily_stats[d]['min'] for d in dates]
maxs = [daily_stats[d]['max'] for d in dates]
std = [daily_stats[d]['std']for d in dates]
twentyfive = [daily_stats[d]['25%']for d in dates]
fifty = [daily_stats[d]['50%'] for d in dates]
seventyfive = [daily_stats[d]['75%']for d in dates]

fig_stats = go.Figure()
stat_names = ['Mean', 'Min', 'Max', 'Std', 'Twentyfive', 'Fifty', 'Seventyfive']
stat_values = [means, mins, maxs, std, twentyfive, fifty, seventyfive]

for name, values in zip(stat_names, stat_values):
    fig_stats.add_trace(go.Scatter(x=dates, y=values, mode='lines+markers', name=name))

fig_stats.update_layout(title='Daily Statistics', xaxis_title='Date', yaxis_title='Value')
fig_stats.show()

# Create a DataFrame from daily_stats for easier manipulation
daily_stats_df = pd.DataFrame(daily_stats).T  # Transpose so dates are the index

# Create a figure to compare and show the changes in daily statistics (mean, min, max, etc.) between consecutive days

fig_diff = go.Figure()

# Use daily_stats_df to compute the difference between consecutive days for each statistic
stats_to_plot = ['mean', 'min', 'max', 'std', '25%', '50%', '75%']
for stat in stats_to_plot:
    # Compute the difference between consecutive days
    diff = daily_stats_df[stat].diff().iloc[1:]  # skip the first NaN
    fig_diff.add_trace(
        go.Scatter(
            x=diff.index,
            y=diff.values,
            mode='lines+markers',
            name=f'Δ {stat}'
        )
    )



fig_diff.update_layout(
    title='Change in Daily Statistics Between Consecutive Days',
    xaxis_title='Date',
    yaxis_title='Change in Value'
)
fig_diff.show()




In [2]:
import pandas as pd

import plotly.express as px

import plotly.graph_objects as po

df_printer = pd.read_csv("../../data/raw/printer-data.csv") 
df_water = pd.read_csv("../../data/raw/water-data.csv") 

# Separate the 'time' column into 'date' and 'clock_time' columns
df_printer['date'] = pd.to_datetime(df_printer['time']).dt.date
df_printer['clock_time'] = pd.to_datetime(df_printer['time']).dt.time

df_water['date'] = pd.to_datetime(df_water['time']).dt.date
df_water['clock_time'] = pd.to_datetime(df_water['time']).dt.time
# print(df_printer.head())

# Calculate statistics for both dataframes
stats_printer = df_printer['value'].describe()
stats_water = df_water['value'].describe()

# Combine statistics into a DataFrame for comparison
stats_df = pd.DataFrame({
    'Printer': stats_printer,
    'Water': stats_water
})

import plotly.graph_objects as go

fig_compare = go.Figure()

for col in stats_df.columns:
    fig_compare.add_trace(
        go.Scatter(
            x=stats_df.index,
            y=stats_df[col],
            mode='lines+markers',
            name=col
        )
    )

fig_compare.update_layout(
    title='Comparison of Value Statistics: Printer vs Water',
    xaxis_title='Statistic',
    yaxis_title='Value'
)
fig_compare.show()
# Concatenate the 'value' columns from printer and water in the order: printer, then water
combined_values = pd.concat([df_printer['value'], df_water['value']], ignore_index=True)

# Create a label array to indicate the source of each value
labels = ['Printer'] * len(df_printer) + ['Water'] * len(df_water)

# Create a DataFrame to hold the combined data
combined_df = pd.DataFrame({'Source': labels, 'Value': combined_values})

# Optionally, plot the combined data to visualize the connection/order
fig_combined = px.line(combined_df, y='Value', color='Source', title='Printer and Water Values in Sequence')
fig_combined.show()
# Calculate rolling statistics (mean, min, max) for the combined data
window_size = 50  # You can adjust the window size as needed
combined_df['Rolling Mean'] = combined_df['Value'].rolling(window=window_size, min_periods=1).mean()
combined_df['Rolling Min'] = combined_df['Value'].rolling(window=window_size, min_periods=1).min()
combined_df['Rolling Max'] = combined_df['Value'].rolling(window=window_size, min_periods=1).max()

# Plot the combined values with rolling statistics
fig_stats_combined = go.Figure()
fig_stats_combined.add_trace(go.Scatter(
    y=combined_df['Value'],
    mode='lines',
    name='Value',
    line=dict(color='gray'),
    opacity=0.5
))
fig_stats_combined.add_trace(go.Scatter(
    y=combined_df['Rolling Mean'],
    mode='lines',
    name='Rolling Mean',
    line=dict(color='blue')
))
fig_stats_combined.add_trace(go.Scatter(
    y=combined_df['Rolling Min'],
    mode='lines',
    name='Rolling Min',
    line=dict(color='green', dash='dot')
))
fig_stats_combined.add_trace(go.Scatter(
    y=combined_df['Rolling Max'],
    mode='lines',
    name='Rolling Max',
    line=dict(color='red', dash='dot')
))
fig_stats_combined.update_layout(
    title='Combined Printer and Water Values with Rolling Statistics',
    xaxis_title='Index (Printer then Water)',
    yaxis_title='Value'
)
fig_stats_combined.show()